In [2]:
# ============================================================
# VALIDATED TWO-TRACK SIMULATION: OVERLAP-ADAPTIVE DR ESTIMATOR
#
# Track 1: FIXED_TARGET
# Track 2: ESTIMATED_TARGET
#
# Estimators: OR, IPW (Hajek), and DR
# Full pairs bootstrap per track
#
# ALL OUTPUTS SAVED IN:
# D:\OVERLAP_DR_SIMULATION_RESULTS_4\
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from scipy.special import expit
from scipy.stats import multivariate_normal
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 12345

N_LIST = [1000, 2000, 5000]

A_VALUES = [0.0, 0.005, 0.01, 0.05, 0.10]

N_REP = 500
B_BOOT = 200

N_TRUE = 200000

RHO = 0.50
EPS = 1e-5

SCENARIOS = [
    "ALL_CORRECT",
    "OUTCOME_MISSPECIFIED",
    "NUISANCE_MISSPECIFIED",
    "BOTH_MISSPECIFIED"
]

TRACKS = {
    "FIXED_TARGET": SCENARIOS,
    "ESTIMATED_TARGET": [
        "ALL_CORRECT",
        "OUTCOME_MISSPECIFIED"
    ]
}

np.random.seed(SEED)


# ============================================================
# 2. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    r"D:\OVERLAP_DR_SIMULATION_RESULTS_4"
)

CSV_DIR = OUTPUT_DIR / "CSV_Results"

PLOT_DIR = OUTPUT_DIR / "Plots"

LOG_DIR = OUTPUT_DIR / "Logs"

for folder in [OUTPUT_DIR, CSV_DIR, PLOT_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not Path("D:/").exists():
    raise FileNotFoundError(
        "D drive was not found. Please check your D drive."
    )

print("\nOUTPUT DIRECTORY")
print("================")
print(OUTPUT_DIR)


# ============================================================
# 3. SOURCE AND TARGET DISTRIBUTIONS
# ============================================================

MU_SOURCE = np.array([0.5, -0.4])

MU_TARGET = np.array([0.0, 0.0])

SIGMA_SOURCE = np.array([
    [1.2, 0.2],
    [0.2, 0.8]
])

SIGMA_TARGET = np.array([
    [0.8, 0.1],
    [0.1, 1.1]
])


# ============================================================
# 4. TRUE DATA-GENERATING MODELS
# ============================================================

def true_e(x):

    log_fs = multivariate_normal.logpdf(
        x,
        mean=MU_SOURCE,
        cov=SIGMA_SOURCE
    )

    log_ft = multivariate_normal.logpdf(
        x,
        mean=MU_TARGET,
        cov=SIGMA_TARGET
    )

    logit_e = (
        np.log(RHO / (1 - RHO))
        + log_fs - log_ft
    )

    return expit(logit_e)


def true_p(x):

    x1 = x[:, 0]
    x2 = x[:, 1]

    return expit(
        -0.2
        + 0.8 * x1
        - 0.5 * x2
        + 0.35 * x1 * x2
        + 0.2 * x1**2
    )


def true_r(g, x):

    x1 = x[:, 0]
    x2 = x[:, 1]

    return expit(
        -0.3
        + 0.3 * x1
        - 0.2 * x2
        + 0.2 * g
        + 0.25 * x1 * x2
        - 0.1 * x2**2
    )


def true_mu0(x):

    x1 = x[:, 0]
    x2 = x[:, 1]

    return (
        1.0
        + 0.5 * x1
        - 0.3 * x2
        + 0.25 * x1**2
    )


def true_effect(x):

    x1 = x[:, 0]
    x2 = x[:, 1]

    return (
        0.7
        + 0.4 * x1
        - 0.2 * x2
        + 0.15 * x1 * x2
    )


def true_mu(g, x):

    return true_mu0(x) + g * true_effect(x)


# ============================================================
# 5. OVERLAP FUNCTIONS
# ============================================================

def true_tau(x):

    e = np.clip(
        true_e(x),
        EPS,
        1 - EPS
    )

    p = np.clip(
        true_p(x),
        EPS,
        1 - EPS
    )

    r1 = np.clip(
        true_r(1, x),
        EPS,
        1 - EPS
    )

    r0 = np.clip(
        true_r(0, x),
        EPS,
        1 - EPS
    )

    tau1 = e * r1 * p

    tau0 = e * r0 * (1 - p)

    return (
        tau1 * tau0
        / np.maximum(tau1 + tau0, EPS)
    )


def estimated_tau(models):

    e = np.clip(
        models["e"],
        EPS,
        1 - EPS
    )

    p = np.clip(
        models["p"],
        EPS,
        1 - EPS
    )

    r1 = np.clip(
        models["r1"],
        EPS,
        1 - EPS
    )

    r0 = np.clip(
        models["r0"],
        EPS,
        1 - EPS
    )

    tau1 = e * r1 * p

    tau0 = e * r0 * (1 - p)

    return (
        tau1 * tau0
        / np.maximum(tau1 + tau0, EPS)
    )


def overlap_phi(tau, a):

    tau = np.clip(
        np.asarray(tau),
        0.0,
        None
    )

    if a == 0:

        return np.ones_like(
            tau,
            dtype=float
        )

    return (
        tau
        / np.maximum(tau + a, EPS)
    )


# ============================================================
# 6. DATA GENERATION
# ============================================================

def generate_data(n, seed):

    rng = np.random.default_rng(seed)

    Z = rng.binomial(
        1,
        RHO,
        size=n
    )

    X = np.zeros((n, 2))

    source_idx = np.where(Z == 1)[0]

    target_idx = np.where(Z == 0)[0]

    X[source_idx] = rng.multivariate_normal(
        MU_SOURCE,
        SIGMA_SOURCE,
        size=len(source_idx)
    )

    X[target_idx] = rng.multivariate_normal(
        MU_TARGET,
        SIGMA_TARGET,
        size=len(target_idx)
    )

    W = np.full(
        n,
        -1,
        dtype=int
    )

    W[source_idx] = rng.binomial(
        1,
        true_p(X[source_idx])
    )

    S = np.zeros(
        n,
        dtype=int
    )

    for g in [0, 1]:

        arm_idx = source_idx[
            W[source_idx] == g
        ]

        if len(arm_idx) > 0:

            S[arm_idx] = rng.binomial(
                1,
                true_r(g, X[arm_idx])
            )

    Y = np.full(
        n,
        np.nan
    )

    observed_idx = np.where(
        (Z == 1) & (S == 1)
    )[0]

    if len(observed_idx) > 0:

        Y[observed_idx] = (
            true_mu(
                W[observed_idx],
                X[observed_idx]
            )
            + rng.normal(
                0,
                1,
                len(observed_idx)
            )
        )

    return {
        "X": X,
        "Z": Z,
        "W": W,
        "S": S,
        "Y": Y
    }


# ============================================================
# 7. DESIGN MATRICES
# ============================================================

def design_matrix(x, correct=True):

    x1 = x[:, 0]

    x2 = x[:, 1]

    if correct:

        return np.column_stack([
            np.ones(len(x)),
            x1,
            x2,
            x1**2,
            x2**2,
            x1 * x2
        ])

    return np.column_stack([
        np.ones(len(x)),
        x1,
        x2
    ])


# ============================================================
# 8. LOGISTIC REGRESSION HELPERS
# ============================================================

def fit_logistic(X, y):

    y = np.asarray(y).astype(int)

    classes = np.unique(y)

    if len(classes) == 1:

        return (
            "constant",
            np.clip(
                float(classes[0]),
                EPS,
                1 - EPS
            )
        )

    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=3000,
        fit_intercept=False
    )

    model.fit(X, y)

    return (
        "model",
        model
    )


def predict_logistic(fitted, X):

    kind, model = fitted

    if kind == "constant":

        return np.full(
            len(X),
            model
        )

    return model.predict_proba(X)[:, 1]


# ============================================================
# 9. NUISANCE MODELS
# ============================================================

def fit_nuisance(data, correct=True):

    X = data["X"]

    Z = data["Z"]

    W = data["W"]

    S = data["S"]

    F = design_matrix(
        X,
        correct
    )

    source = np.where(Z == 1)[0]

    if len(source) == 0:

        raise ValueError(
            "No source observations."
        )

    # Selection model
    e_model = fit_logistic(
        F,
        Z
    )

    e_hat = predict_logistic(
        e_model,
        F
    )

    # Treatment model
    p_model = fit_logistic(
        F[source],
        W[source]
    )

    p_hat = predict_logistic(
        p_model,
        F
    )

    # Missingness models
    r_hat = {}

    for g in [0, 1]:

        arm = source[
            W[source] == g
        ]

        if len(arm) == 0:

            raise ValueError(
                f"No source subjects in arm {g}."
            )

        r_model = fit_logistic(
            F[arm],
            S[arm]
        )

        r_hat[g] = predict_logistic(
            r_model,
            F
        )

    return {
        "e": np.clip(e_hat, EPS, 1 - EPS),
        "p": np.clip(p_hat, EPS, 1 - EPS),
        "r1": np.clip(r_hat[1], EPS, 1 - EPS),
        "r0": np.clip(r_hat[0], EPS, 1 - EPS)
    }


# ============================================================
# 10. OUTCOME REGRESSION
# ============================================================

def fit_outcome(data, correct=True):

    X = data["X"]

    Z = data["Z"]

    W = data["W"]

    S = data["S"]

    Y = data["Y"]

    F = design_matrix(
        X,
        correct
    )

    mu_hat = {}

    for g in [0, 1]:

        idx = np.where(
            (Z == 1)
            & (W == g)
            & (S == 1)
            & np.isfinite(Y)
        )[0]

        if len(idx) < 2:

            raise ValueError(
                f"Insufficient observed outcomes in arm {g}."
            )

        beta = np.linalg.lstsq(
            F[idx],
            Y[idx],
            rcond=None
        )[0]

        mu_hat[g] = F @ beta

    return mu_hat


# ============================================================
# 11. FIT MODELS FOR FOUR SCENARIOS
# ============================================================

def fit_models(data, scenario):

    correct_outcome = scenario in [
        "ALL_CORRECT",
        "NUISANCE_MISSPECIFIED"
    ]

    correct_nuisance = scenario in [
        "ALL_CORRECT",
        "OUTCOME_MISSPECIFIED"
    ]

    nuisance = fit_nuisance(
        data,
        correct_nuisance
    )

    outcome = fit_outcome(
        data,
        correct_outcome
    )

    return {
        **nuisance,
        "mu": outcome
    }


# ============================================================
# 12. OR, IPW (HAJEK), AND DR ESTIMATORS
# ============================================================

def estimate_effects(
    data,
    models,
    overlap_mode="FIXED_TARGET"
):

    X = data["X"]

    Z = data["Z"]

    W = data["W"]

    S = data["S"]

    Y = data["Y"]

    e = np.clip(
        models["e"],
        EPS,
        1 - EPS
    )

    p = np.clip(
        models["p"],
        EPS,
        1 - EPS
    )

    r1 = np.clip(
        models["r1"],
        EPS,
        1 - EPS
    )

    r0 = np.clip(
        models["r0"],
        EPS,
        1 - EPS
    )

    mu1 = models["mu"][1]

    mu0 = models["mu"][0]

    target = (Z == 0)

    source_observed_1 = (
        (Z == 1)
        & (S == 1)
        & (W == 1)
    )

    source_observed_0 = (
        (Z == 1)
        & (S == 1)
        & (W == 0)
    )

    omega1 = np.zeros(
        len(X),
        dtype=float
    )

    omega0 = np.zeros(
        len(X),
        dtype=float
    )

    omega1[source_observed_1] = (
        (1 - e[source_observed_1])
        /
        (
            e[source_observed_1]
            * r1[source_observed_1]
            * p[source_observed_1]
        )
    )

    omega0[source_observed_0] = (
        (1 - e[source_observed_0])
        /
        (
            e[source_observed_0]
            * r0[source_observed_0]
            * (1 - p[source_observed_0])
        )
    )

    omega1 = np.nan_to_num(
        omega1,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    omega0 = np.nan_to_num(
        omega0,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    y_safe = np.where(
        np.isfinite(Y),
        Y,
        0.0
    )

    resid1 = np.where(
        source_observed_1,
        y_safe - mu1,
        0.0
    )

    resid0 = np.where(
        source_observed_0,
        y_safe - mu0,
        0.0
    )

    # Overlap target definition
    if overlap_mode == "FIXED_TARGET":

        tau_for_phi = true_tau(X)

    elif overlap_mode == "ESTIMATED_TARGET":

        tau_for_phi = estimated_tau(models)

    else:

        raise ValueError(
            f"Unknown overlap_mode: {overlap_mode}"
        )

    results = {}

    for a in A_VALUES:

        phi = overlap_phi(
            tau_for_phi,
            a
        )

        denominator = np.sum(
            target * phi
        )

        if denominator <= EPS:

            raise ValueError(
                "Estimated target overlap denominator is zero."
            )

        # Outcome regression estimator
        or_estimate = (
            np.sum(
                phi[target]
                * (mu1[target] - mu0[target])
            )
            / denominator
        )

        # Doubly robust estimator
        numerator1_dr = (
            np.sum(target * phi * mu1)
            + np.sum(phi * omega1 * resid1)
        )

        numerator0_dr = (
            np.sum(target * phi * mu0)
            + np.sum(phi * omega0 * resid0)
        )

        dr_estimate = (
            numerator1_dr - numerator0_dr
        ) / denominator

        # Hajek IPW estimator
        weighted_den1 = np.sum(
            phi * omega1
        )

        weighted_den0 = np.sum(
            phi * omega0
        )

        if (
            weighted_den1 <= EPS
            or weighted_den0 <= EPS
        ):

            raise ValueError(
                "IPW weighted denominator is zero."
            )

        ipw_mean1 = (
            np.sum(phi * omega1 * y_safe)
            / weighted_den1
        )

        ipw_mean0 = (
            np.sum(phi * omega0 * y_safe)
            / weighted_den0
        )

        ipw_estimate = (
            ipw_mean1 - ipw_mean0
        )

        # Effective sample sizes
        aw1 = phi * omega1

        aw0 = phi * omega0

        ess1 = (
            np.sum(aw1) ** 2
            / max(np.sum(aw1 ** 2), EPS)
        )

        ess0 = (
            np.sum(aw0) ** 2
            / max(np.sum(aw0 ** 2), EPS)
        )

        estimates = {
            "OR": or_estimate,
            "IPW": ipw_estimate,
            "DR": dr_estimate
        }

        if not all(
            np.isfinite(v)
            for v in estimates.values()
        ):

            raise ValueError(
                "At least one estimator returned a non-finite value."
            )

        results[a] = {
            **estimates,
            "ESS1": ess1,
            "ESS0": ess0,
            "Mean_Phi_Target": float(
                np.mean(phi[target])
            ),
            "Mean_Tau_Target": float(
                np.mean(tau_for_phi[target])
            )
        }

    return results


# ============================================================
# 13. TRUE OVERLAP-TARGET ATE
# ============================================================

def calculate_true_values():

    rng = np.random.default_rng(
        SEED + 999
    )

    X_target = rng.multivariate_normal(
        MU_TARGET,
        SIGMA_TARGET,
        size=N_TRUE
    )

    effect = true_effect(
        X_target
    )

    truth = {}

    for a in A_VALUES:

        phi = overlap_phi(
            true_tau(X_target),
            a
        )

        truth[a] = (
            np.sum(phi * effect)
            / np.sum(phi)
        )

    return truth


# ============================================================
# 14. ORDINARY PAIRS BOOTSTRAP
# ============================================================

def bootstrap_sample(data, rng):

    n = len(data["Z"])

    idx = rng.integers(
        0,
        n,
        size=n
    )

    return {
        key: value[idx]
        for key, value in data.items()
    }


# ============================================================
# 15. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def calculate_bootstrap_cis(
    data,
    scenario,
    B,
    seed,
    overlap_mode="FIXED_TARGET"
):

    rng = np.random.default_rng(
        seed
    )

    boot_values = {
        (estimator, a): []
        for estimator in ["OR", "IPW", "DR"]
        for a in A_VALUES
    }

    failures = 0

    error_messages = []

    for b in range(B):

        try:

            boot_data = bootstrap_sample(
                data,
                rng
            )

            boot_models = fit_models(
                boot_data,
                scenario
            )

            boot_estimates = estimate_effects(
                boot_data,
                boot_models,
                overlap_mode=overlap_mode
            )

            for a in A_VALUES:

                for estimator in ["OR", "IPW", "DR"]:

                    value = boot_estimates[a][estimator]

                    if np.isfinite(value):

                        boot_values[
                            (estimator, a)
                        ].append(value)

        except Exception as error:

            failures += 1

            if len(error_messages) < 3:

                error_messages.append(
                    repr(error)
                )

    ci = {}

    min_success = max(
        10,
        int(np.ceil(0.80 * B))
    )

    for estimator in ["OR", "IPW", "DR"]:

        for a in A_VALUES:

            values = np.asarray(
                boot_values[
                    (estimator, a)
                ]
            )

            if len(values) < min_success:

                ci[(estimator, a)] = {
                    "lower": np.nan,
                    "upper": np.nan,
                    "length": np.nan,
                    "success": len(values)
                }

            else:

                lower, upper = np.quantile(
                    values,
                    [0.025, 0.975]
                )

                ci[(estimator, a)] = {
                    "lower": lower,
                    "upper": upper,
                    "length": upper - lower,
                    "success": len(values)
                }

    return (
        ci,
        failures,
        error_messages
    )


# ============================================================
# 16. ONE MONTE CARLO REPLICATION
# ============================================================

def run_replication(
    n,
    scenario,
    seed,
    truth,
    overlap_mode="FIXED_TARGET"
):

    data = generate_data(
        n,
        seed
    )

    models = fit_models(
        data,
        scenario
    )

    estimates = estimate_effects(
        data,
        models,
        overlap_mode=overlap_mode
    )

    ci, failures, error_messages = calculate_bootstrap_cis(
        data,
        scenario,
        B_BOOT,
        seed + 500000,
        overlap_mode=overlap_mode
    )

    records = []

    for a in A_VALUES:

        for estimator in ["OR", "IPW", "DR"]:

            estimate = estimates[a][estimator]

            lower = ci[
                (estimator, a)
            ]["lower"]

            upper = ci[
                (estimator, a)
            ]["upper"]

            coverage = (
                float(
                    lower <= truth[a] <= upper
                )
                if np.isfinite(lower)
                and np.isfinite(upper)
                else np.nan
            )

            records.append({

                "Track": overlap_mode,

                "N": n,

                "Scenario": scenario,

                "Estimator": estimator,

                "a": a,

                "True_ATE": truth[a],

                "Estimate": estimate,

                "Bias": estimate - truth[a],

                "CI_Lower": lower,

                "CI_Upper": upper,

                "Coverage": coverage,

                "CI_Length": ci[
                    (estimator, a)
                ]["length"],

                "ESS1": estimates[a]["ESS1"],

                "ESS0": estimates[a]["ESS0"],

                "Mean_Phi_Target": estimates[a][
                    "Mean_Phi_Target"
                ],

                "Mean_Tau_Target": estimates[a][
                    "Mean_Tau_Target"
                ],

                "Successful_Bootstraps": ci[
                    (estimator, a)
                ]["success"],

                "Failed_Bootstraps": failures

            })

    return records, error_messages


# ============================================================
# 17. MAIN SIMULATION
# ============================================================

def run_simulation():

    truth = calculate_true_values()

    print("\nTRUE DGP OVERLAP-TARGET ATE")

    print("===========================")

    for a in A_VALUES:

        print(
            f"a={a:.3f}: {truth[a]:.6f}"
        )

    all_records = []

    bootstrap_errors = []

    total = (
        sum(
            len(v)
            for v in TRACKS.values()
        )
        * len(N_LIST)
        * N_REP
    )

    progress = tqdm(
        total=total,
        desc="Monte Carlo simulation"
    )

    for overlap_mode, scenarios_for_track in TRACKS.items():

        print(
            f"\n\n========== TRACK: {overlap_mode} =========="
        )

        for n in N_LIST:

            for scenario in scenarios_for_track:

                print(
                    f"\nTrack={overlap_mode}, "
                    f"N={n}, Scenario={scenario}"
                )

                for rep in range(N_REP):

                    seed = (
                        SEED
                        + n * 100000
                        + SCENARIOS.index(scenario) * 10000
                        + rep
                        + (
                            0
                            if overlap_mode == "FIXED_TARGET"
                            else 7000000
                        )
                    )

                    try:

                        records, errors = run_replication(
                            n,
                            scenario,
                            seed,
                            truth,
                            overlap_mode=overlap_mode
                        )

                        all_records.extend(
                            records
                        )

                        if errors:

                            bootstrap_errors.extend([

                                {
                                    "Track": overlap_mode,

                                    "N": n,

                                    "Scenario": scenario,

                                    "Replication": rep,

                                    "Error": msg
                                }

                                for msg in errors

                            ])

                    except Exception as error:

                        progress.close()

                        raise RuntimeError(
                            f"Replication failed: "
                            f"Track={overlap_mode}, "
                            f"N={n}, "
                            f"Scenario={scenario}, "
                            f"Replication={rep}: "
                            f"{repr(error)}"
                        ) from error

                    progress.update(1)

    progress.close()

    # --------------------------------------------------------
    # SAVE RAW RESULTS
    # --------------------------------------------------------

    raw = pd.DataFrame(
        all_records
    )

    raw_file = (
        CSV_DIR
        / "simulation_replication_results.csv"
    )

    raw.to_csv(
        raw_file,
        index=False
    )

    print(
        "\nRaw simulation results saved:"
    )

    print(raw_file)

    # --------------------------------------------------------
    # SAVE BOOTSTRAP ERROR LOG
    # --------------------------------------------------------

    # IMPORTANT FIX:
    # Explicit columns ensure that even an empty error log
    # contains CSV headers. Therefore pd.read_csv() will work.

    error_columns = [
        "Track",
        "N",
        "Scenario",
        "Replication",
        "Error"
    ]

    error_df = pd.DataFrame(
        bootstrap_errors,
        columns=error_columns
    )

    error_file = (
        LOG_DIR
        / "bootstrap_error_log.csv"
    )

    error_df.to_csv(
        error_file,
        index=False
    )

    print(
        "\nBootstrap error log saved:"
    )

    print(error_file)

    # --------------------------------------------------------
    # SUMMARY STATISTICS
    # --------------------------------------------------------

    summary = []

    group_cols = [
        "Track",
        "N",
        "Scenario",
        "Estimator",
        "a"
    ]

    for keys, group in raw.groupby(
        group_cols
    ):

        track, n, scenario, estimator, a = keys

        estimates = group[
            "Estimate"
        ].dropna()

        truth_value = group[
            "True_ATE"
        ].iloc[0]

        if len(estimates) == 0:

            continue

        summary.append({

            "Track": track,

            "N": n,

            "Scenario": scenario,

            "Estimator": estimator,

            "a": a,

            "True_ATE": truth_value,

            "Mean_Estimate": estimates.mean(),

            "Bias": estimates.mean() - truth_value,

            "Empirical_SD": estimates.std(ddof=1),

            "RMSE": np.sqrt(
                np.mean(
                    (estimates - truth_value) ** 2
                )
            ),

            "Coverage": group[
                "Coverage"
            ].mean(),

            "Average_CI_Length": group[
                "CI_Length"
            ].mean(),

            "Average_ESS1": group[
                "ESS1"
            ].mean(),

            "Average_ESS0": group[
                "ESS0"
            ].mean(),

            "Average_Mean_Phi_Target": group[
                "Mean_Phi_Target"
            ].mean(),

            "Average_Mean_Tau_Target": group[
                "Mean_Tau_Target"
            ].mean(),

            "Successful_Replications": len(
                estimates
            ),

            "Average_Successful_Bootstraps": group[
                "Successful_Bootstraps"
            ].mean(),

            "Average_Failed_Bootstraps": group[
                "Failed_Bootstraps"
            ].mean()

        })

    summary_df = pd.DataFrame(
        summary
    )

    # --------------------------------------------------------
    # SAVE SUMMARY CSV
    # --------------------------------------------------------

    summary_file = (
        CSV_DIR
        / "simulation_summary_results.csv"
    )

    summary_df.to_csv(
        summary_file,
        index=False
    )

    print(
        "\nSummary results saved:"
    )

    print(summary_file)

    print(
        "\nFINAL SIMULATION SUMMARY"
    )

    print(
        "========================"
    )

    print(
        summary_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    return raw, summary_df


# ============================================================
# 18. PLOTS
# ============================================================

def make_plots(summary):

    if summary.empty:

        print(
            "No results available for plotting."
        )

        return

    for track in TRACKS:

        for estimator in ["OR", "IPW", "DR"]:

            for scenario in TRACKS[track]:

                for n in N_LIST:

                    d = summary[

                        (summary["Track"] == track)

                        & (
                            summary["Estimator"]
                            == estimator
                        )

                        & (
                            summary["Scenario"]
                            == scenario
                        )

                        & (
                            summary["N"] == n
                        )

                    ].sort_values("a")

                    if d.empty:

                        continue

                    # ------------------------------------------------
                    # BIAS PLOT
                    # ------------------------------------------------

                    plt.figure(
                        figsize=(7, 5)
                    )

                    plt.plot(
                        d["a"],
                        d["Bias"],
                        marker="o"
                    )

                    plt.axhline(
                        0,
                        linestyle="--"
                    )

                    plt.xlabel(
                        "Overlap tuning parameter (a)"
                    )

                    plt.ylabel(
                        "Bias"
                    )

                    plt.title(
                        f"{track} - {estimator} Bias: "
                        f"{scenario}, N={n}"
                    )

                    plt.tight_layout()

                    bias_file = (
                        PLOT_DIR
                        /
                        f"{track}_{estimator}_Bias_{scenario}_N{n}.png"
                    )

                    plt.savefig(
                        bias_file,
                        dpi=300,
                        bbox_inches="tight"
                    )

                    plt.close()

                    # ------------------------------------------------
                    # RMSE PLOT
                    # ------------------------------------------------

                    plt.figure(
                        figsize=(7, 5)
                    )

                    plt.plot(
                        d["a"],
                        d["RMSE"],
                        marker="o"
                    )

                    plt.xlabel(
                        "Overlap tuning parameter (a)"
                    )

                    plt.ylabel(
                        "RMSE"
                    )

                    plt.title(
                        f"{track} - {estimator} RMSE: "
                        f"{scenario}, N={n}"
                    )

                    plt.tight_layout()

                    rmse_file = (
                        PLOT_DIR
                        /
                        f"{track}_{estimator}_RMSE_{scenario}_N{n}.png"
                    )

                    plt.savefig(
                        rmse_file,
                        dpi=300,
                        bbox_inches="tight"
                    )

                    plt.close()

                    # ------------------------------------------------
                    # COVERAGE PLOT
                    # ------------------------------------------------

                    plt.figure(
                        figsize=(7, 5)
                    )

                    plt.plot(
                        d["a"],
                        d["Coverage"],
                        marker="o"
                    )

                    plt.axhline(
                        0.95,
                        linestyle="--"
                    )

                    plt.ylim(
                        0,
                        1.05
                    )

                    plt.xlabel(
                        "Overlap tuning parameter (a)"
                    )

                    plt.ylabel(
                        "Coverage probability"
                    )

                    plt.title(
                        f"{track} - {estimator} Coverage: "
                        f"{scenario}, N={n}"
                    )

                    plt.tight_layout()

                    coverage_file = (
                        PLOT_DIR
                        /
                        f"{track}_{estimator}_Coverage_{scenario}_N{n}.png"
                    )

                    plt.savefig(
                        coverage_file,
                        dpi=300,
                        bbox_inches="tight"
                    )

                    plt.close()

                    print(
                        f"Saved plots for: "
                        f"{track}, {estimator}, "
                        f"{scenario}, N={n}"
                    )


# ============================================================
# 19. EXECUTE SIMULATION
# ============================================================

if __name__ == "__main__":

    print("\n")
    print("================================================")
    print("OVERLAP-ADAPTIVE DR SIMULATION STARTED")
    print("================================================")

    raw_results, summary_results = run_simulation()

    print("\nGenerating plots...")

    make_plots(
        summary_results
    )

    print("\n")
    print("================================================")
    print("SIMULATION COMPLETED SUCCESSFULLY")
    print("================================================")

    print("\nALL OUTPUTS SAVED IN:")

    print(OUTPUT_DIR)

    print("\n1. RAW SIMULATION RESULTS:")

    print(
        CSV_DIR
        / "simulation_replication_results.csv"
    )

    print("\n2. SUMMARY RESULTS:")

    print(
        CSV_DIR
        / "simulation_summary_results.csv"
    )

    print("\n3. BOOTSTRAP ERROR LOG:")

    print(
        LOG_DIR
        / "bootstrap_error_log.csv"
    )

    print("\n4. ALL PLOTS:")

    print(PLOT_DIR)

    print("\nTotal simulation records:")

    print(len(raw_results))

    print("\nTotal summary records:")

    print(len(summary_results))

    # --------------------------------------------------------
    # SAFE BOOTSTRAP ERROR COUNT
    # --------------------------------------------------------

    error_file = LOG_DIR / "bootstrap_error_log.csv"

    if error_file.exists() and error_file.stat().st_size > 0:

        error_df = pd.read_csv(error_file)

        total_bootstrap_errors = len(error_df)

    else:

        total_bootstrap_errors = 0

    print("\nTotal bootstrap errors:")

    print(total_bootstrap_errors)

    print("\nAll tasks completed!")


OUTPUT DIRECTORY
D:\OVERLAP_DR_SIMULATION_RESULTS_4


OVERLAP-ADAPTIVE DR SIMULATION STARTED

TRUE DGP OVERLAP-TARGET ATE
a=0.000: 0.712552
a=0.005: 0.764238
a=0.010: 0.780243
a=0.050: 0.817659
a=0.100: 0.829497


Monte Carlo simulation:   0%|          | 0/9000 [00:00<?, ?it/s]



========== TRACK: FIXED_TARGET ==========

Track=FIXED_TARGET, N=1000, Scenario=ALL_CORRECT

Track=FIXED_TARGET, N=1000, Scenario=OUTCOME_MISSPECIFIED

Track=FIXED_TARGET, N=1000, Scenario=NUISANCE_MISSPECIFIED

Track=FIXED_TARGET, N=1000, Scenario=BOTH_MISSPECIFIED

Track=FIXED_TARGET, N=2000, Scenario=ALL_CORRECT

Track=FIXED_TARGET, N=2000, Scenario=OUTCOME_MISSPECIFIED

Track=FIXED_TARGET, N=2000, Scenario=NUISANCE_MISSPECIFIED

Track=FIXED_TARGET, N=2000, Scenario=BOTH_MISSPECIFIED

Track=FIXED_TARGET, N=5000, Scenario=ALL_CORRECT

Track=FIXED_TARGET, N=5000, Scenario=OUTCOME_MISSPECIFIED

Track=FIXED_TARGET, N=5000, Scenario=NUISANCE_MISSPECIFIED

Track=FIXED_TARGET, N=5000, Scenario=BOTH_MISSPECIFIED


========== TRACK: ESTIMATED_TARGET ==========

Track=ESTIMATED_TARGET, N=1000, Scenario=ALL_CORRECT

Track=ESTIMATED_TARGET, N=1000, Scenario=OUTCOME_MISSPECIFIED

Track=ESTIMATED_TARGET, N=2000, Scenario=ALL_CORRECT

Track=ESTIMATED_TARGET, N=2000, Scenario=OUTCOME_MISSPECIFIED